## MarketWatch Scraper

In [11]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import duckdb as dd
from datetime import datetime
import time


DB_PATH = "marketwatch_news.duckdb"
headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept-Encoding': 'gzip, deflate, br',
        'Origin': 'https://www.marketwatch.com',
        'Referer': 'https://www.marketwatch.com/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-User': '?1',
        'Sec-Fetch-Dest': 'document',
        'Sec-CH-UA': '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        'Sec-CH-UA-Mobile': '?0',
        'Sec-CH-UA-Platform': '"Windows"',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
    }
delay_seconds = float(1.5)
session = requests.Session()
session.headers.update(headers)
_session_primed = False

def init_db(db_path: str = DB_PATH):
    conn = dd.connect(db_path)
    conn.execute(
        '''
        DROP TABLE IF EXISTS marketwatch_articles;

        CREATE TABLE IF NOT EXISTS marketwatch_articles (
            query TEXT,
            title TEXT,
            url TEXT,
            subhead TEXT,
            snippet TEXT,
            author TEXT,
            published_at TIMESTAMP,
            last_updated_at TIMESTAMP,
            scraped_at TIMESTAMP
        );
        '''
    )
    conn.close()


def build_search_url(query: str) -> str:
    """
    Use MarketWatch search, sorted by recency.
    This targets both tickers and company names.
    """
    q = quote_plus(query)
    # Basic search URL; you can refine with more params if needed
    ## return f"https://www.marketwatch.com/search?q={q}&page={page}&sort=recency"
    return f"https://www.marketwatch.com/search?q={q}&ts=2&m=0&sm=0&tab=All%20News"


def prime_session():
    global _session_primed
    if _session_primed:
        return
    try:
        resp = session.get("https://www.marketwatch.com/", timeout=15)
        resp.raise_for_status()
    except requests.RequestException:
        pass
    _session_primed = True


def fetch_search_page(query: str) -> BeautifulSoup:
    global session, _session_primed
    url = build_search_url(query)
    prime_session()

    # Use the global session so cookies and headers persist between homepage
    # requests and search requests. This reduces the chance of a 401 Forbidden.
    session.headers.update({
        "Referer": "https://www.marketwatch.com/",
        "Origin": "https://www.marketwatch.com",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-User": "?1",
        "Sec-Fetch-Dest": "document",
        "Sec-CH-UA": '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        "Sec-CH-UA-Mobile": "?0",
        "Sec-CH-UA-Platform": '"Windows"',
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
    })

    try:
        session.get("https://www.marketwatch.com/", timeout=15)
    except requests.RequestException:
        pass

    resp = session.get(url, timeout=15, allow_redirects=True)
    if resp.status_code == 401:
        # If the site still returns 401, reset the global session and retry
        # with a fresh browser-like session and homepage cookies.
        session = requests.Session()
        session.headers.update(headers)
        _session_primed = False
        prime_session()
        try:
            session.get("https://www.marketwatch.com/", timeout=15)
        except requests.RequestException:
            pass
        resp = session.get(url, timeout=15, allow_redirects=True)

    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def fetch_article_page(url: str) -> BeautifulSoup:
    prime_session()
    if not url.startswith("http"):
        url = f"https://www.marketwatch.com{url}"
    session.headers.update({"Referer": "https://www.marketwatch.com/"})
    resp = session.get(url, timeout=15, allow_redirects=True)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def parse_search_results(soup: BeautifulSoup, delay_seconds: float = delay_seconds):
    """
    Parse MarketWatch search results and article pages.

    NOTE: MarketWatch may change its HTML structure.
    Inspect the page in your browser and adjust selectors if needed.
    """
    articles = []

    # Common pattern: result cards under something like:
    # <div class="searchresult"> or <div class="article__content"> etc.
    # We'll try a couple of patterns.
    result_containers = soup.select("div.searchresult, div.article__content")

    for container in result_containers:
        # Title and URL
        a = container.find("a", href=True)
        if not a:
            continue

        title = a.get_text(strip=True)
        url = str(a["href"])
        subhead = None
        snippet = None
        author = None
        published_at = None
        last_updated_at = None
        try:
            article_page = fetch_article_page(url)

            # Subhead
            subhead_el = article_page.select_one('h2[class*="article__subhead"]')
            subhead = subhead_el.get_text(strip=True) if subhead_el else None

            # Snippet
            snippet_el = article_page.select_one('section[class*="Container"]')
            snippet = snippet_el.get_text(strip=True) if snippet_el else None

            # Author
            author_el = article_page.select_one('a[data-testid*="author-link"]')
            author = author_el.get_text(strip=True) if author_el else None

            #Published Timestamp
            published_time_el = article_page.select_one('div[class*="article__timestamp"]>span[class*="first"]>time')
            published_at = published_time_el['datetime'] if published_time_el else None

            #Last Updated Timestamp
            last_updated_time_el = article_page.select_one('div[class*="article__timestamp"]>span[class*="last"]>time')
            last_updated_at = last_updated_time_el['datetime'] if last_updated_time_el else None
        
        except:
            # If fetching/parsing article page fails, we can still save basic info
            pass

        articles.append(
            {
                "title": title,
                "url": url,
                "subhead": subhead,
                "snippet": snippet,
                "author": author,
                "published_at": published_at,
                "last_updated_at": last_updated_at
            }
        )

        time.sleep(delay_seconds)  # be polite

    return articles


def save_articles_to_duckdb(
    query: str,
    articles,
    db_path: str = DB_PATH,
):
    conn = dd.connect(db_path)
    scraped_at = datetime.utcnow()
    
    for art in articles:
        conn.execute(
            """
            INSERT INTO marketwatch_articles
                (query, title, url, subhead, snippet, author, published_at, last_updated_at, scraped_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            [
                query,
                art.get("title"),
                art.get("url"),
                art.get("subhead"),
                art.get("snippet"),
                art.get("author"),
                art.get("published_at"),
                art.get("last_updated_at"),
                scraped_at,
            ],
        )

    conn.close()


def scrape_marketwatch_for_query(
    query: str,
    pages: int = 1,
    delay_seconds: float = delay_seconds,
):
    """
    Scrape MarketWatch search results for a ticker or company name
    and store them into DuckDB.
    """
    all_articles = []

    for page in range(1, pages + 1):
        print(f"Fetching page {page} for query '{query}'...")
        soup = fetch_search_page(query)
        articles = parse_search_results(soup)
        print(f"  Found {len(articles)} articles on page {page}.")
        if articles not in all_articles:
            all_articles.extend(articles)
        time.sleep(delay_seconds)  # be polite
    
    save_articles_to_duckdb(query, all_articles)  
    print(f"Saved {len(all_articles)} articles for query '{query}'.")


## Run Main Program

In [12]:
if __name__ == "__main__":
    # 1. Initialize DB (run once)
    init_db()

    # 2. Example: scrape daily news for a ticker
    #    You can schedule this script via cron/Task Scheduler to run daily.
    tickers = ['TSLA'] ##, 'MSFT', 'AAPL', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK.A', 'V', 'UNH', 'HD', 'PG', 'MA', 'DIS', 'CMCSA', 'PFE' , 'VZ', 'KO', 'MRK', 'IBM']          # or "MSFT", "TSLA", etc.
    ## company_name = "Microsoft"   # you can also use company names

    # Scrape by ticker
    for ticker in tickers:
        scrape_marketwatch_for_query(ticker)

    # Scrape by company name (optional)
    ## scrape_marketwatch_for_query(company_name, pages=1)

Fetching page 1 for query 'TSLA'...
  Found 77 articles on page 1.


C:\Users\Dsavs\AppData\Local\Temp\ipykernel_14400\2245688637.py:209: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  scraped_at = datetime.utcnow()


Saved 77 articles for query 'TSLA'.


## Query Results

In [13]:
## import duckdb as dd
con = dd.connect(DB_PATH)  # Open the persistent database
result = con.execute('''
            WITH main AS (
            SELECT *,
            coalesce(last_updated_at, published_at) AS article_date
            FROM marketwatch_articles
            )

            SELECT DISTINCT * 
            FROM main
            --WHERE (url LIKE '%barrons.com%' or url LIKE '%marketwatch.com%');         
            ;
            '''
).df()
con.close()
result

,query,title,url,subhead,snippet,author,published_at,last_updated_at,scraped_at,article_date
0,TSLA,Ethereum leads way as large cryptocurrencies p...,https://www.marketwatch.com/data-news/ethereum...,NaN,All large cryptocurrencies were up during U.S....,MarketWatch Automation,2026-04-14 14:00:00,NaT,2026-04-16 18:02:39.513746,2026-04-14 14:00:00
1,TSLA,Intel’s stock hasn’t been this hot in 38 years...,https://www.marketwatch.com/story/intels-stock...,Intel stands to benefit from a CPU boom and a ...,As Intel’s stock rises for the eighth session ...,Britney Nguyen,2026-04-10 16:56:00,NaT,2026-04-16 18:02:39.513746,2026-04-10 16:56:00
2,TSLA,,,NaN,NaN,NaN,NaT,NaT,2026-04-16 18:02:39.513746,NaT
3,TSLA,Intel’s stock has been ‘absolutely on fire.’ N...,https://www.marketwatch.com/story/intels-stock...,The chip maker’s stock has surged on good news...,Intel’s stock has been on a strong run in rece...,Britney Nguyen,2026-04-16 15:56:00,NaT,2026-04-16 18:02:39.513746,2026-04-16 15:56:00
4,TSLA,Why this market rally still has room to run — ...,https://www.marketwatch.com/story/why-this-mar...,Nomura strategist says energy shortages could ...,"The S&P 500 is back in record-setting mode, a...",Barbara Kollmeyer,2026-04-16 10:58:00,2026-04-16 12:51:00,2026-04-16 18:02:39.513746,2026-04-16 12:51:00
5,TSLA,Why two Wall Street titans just turned bullish...,https://www.marketwatch.com/story/why-two-wall...,Citigroup and BlackRock shift back to overweig...,While the human and economic costs continue to...,Barbara Kollmeyer,2026-04-14 10:43:00,2026-04-14 12:43:00,2026-04-16 18:02:39.513746,2026-04-14 12:43:00
6,TSLA,Why ‘Rule of 10’ stocks like Nvidia and Meta a...,https://www.marketwatch.com/story/why-rule-of-...,Higher bond yields have been hitting secular g...,"Oil prices are surging again. Usually, that’s ...",Jamie Chisholm,2026-04-13 11:02:00,2026-04-13 13:31:00,2026-04-16 18:02:39.513746,2026-04-13 13:31:00
7,TSLA,Intel’s stock just had its best 9-day stretch ...,https://www.marketwatch.com/story/intels-stock...,The chip maker’s stock is already richly value...,Intel’s stock has been riding on a wave of goo...,Britney Nguyen,2026-04-13 22:38:00,2026-04-14 01:35:00,2026-04-16 18:02:39.513746,2026-04-14 01:35:00
8,TSLA,Wall Street’s fear gauge just flashed an unusu...,https://www.marketwatch.com/story/wall-streets...,The falling ‘VIX’ is a third sign that the bot...,U.S. crude-oil futures sit near a hundred buck...,Jamie Chisholm,2026-04-10 11:29:00,2026-04-10 13:30:00,2026-04-16 18:02:39.513746,2026-04-10 13:30:00
9,TSLA,Tesla may take another stab at making a cheap ...,https://www.marketwatch.com/story/tesla-may-ta...,A small electric SUV is reportedly in the earl...,"Tesla is reportedly considering adding a new, ...",William Gavin,2026-04-09 17:32:00,NaT,2026-04-16 18:02:39.513746,2026-04-09 17:32:00
